In [3]:
from core.model import Transformer
import yaml
from core.config import Config


from flax import nnx

In [5]:
config = Config()
rngs = nnx.Rngs(config.seed)
model  = Transformer(config=config, rngs=rngs)

In [6]:
model.blocks[0].attention

Attention( # Param: 918,528 (3.7 MB), RngState: 4 (24 B), Total: 918,532 (3.7 MB)
  max_context=512,
  head_size=32,
  n_heads=16,
  dim=512,
  kv_heads=4,
  qkv=Linear( # Param: 393,216 (1.6 MB)
    kernel=Param( # 393,216 (1.6 MB)
      value=Array(shape=(512, 768), dtype=dtype('float32'))
    ),
    bias=None,
    in_features=512,
    out_features=768,
    use_bias=False,
    dtype=None,
    param_dtype=float32,
    precision=None,
    dot_general=<function dot_general at 0x7fb552ff6ac0>,
    promote_dtype=<function promote_dtype at 0x7fb551a272e0>,
    preferred_element_type=None
  ),
  tril=Array(shape=(512, 512), dtype=dtype('bool')),
  o_proj=Linear( # Param: 262,656 (1.1 MB)
    kernel=Param( # 262,144 (1.0 MB)
      value=Array(shape=(512, 512), dtype=dtype('float32'))
    ),
    bias=Param( # 512 (2.0 KB)
      value=Array(shape=(512,), dtype=dtype('float32'))
    ),
    in_features=512,
    out_features=512,
    use_bias=True,
    dtype=None,
    param_dtype=float32,
    pre

In [2]:
import jax.numpy as jnp
import jax

In [10]:
setattr(config, "down_dim_q", 64)
setattr(config, "down_dim_kv", 64)
setattr(config, "mla", True)
setattr(config, "inference", True)
setattr(config, "rope_dim", 64)

In [8]:
import math

In [ ]:
class Attention(nnx.Module):
    def __init__(self, config: Config, rngs: nnx.Rngs):
        self.max_context:int = config.max_context
        self.head_size:int   = config.head_size
        self.n_heads: int    = config.n_heads
        self.dim: int        = config.dim
        self.kv_heads: int   = config.kv_heads if config.kv_heads is not None else self.n_heads
        self.qkv: nnx.Linear = nnx.Linear(self.dim, 
                                          self.dim + 2 * self.kv_heads*self.head_size,
                                          use_bias=False, 
                                          rngs=rngs)
        self.tril: jnp.ndarray  = jnp.tril(
            jnp.ones((self.max_context, self.max_context), dtype=bool)
        )
        self.o_proj: nnx.Linear = nnx.Linear(self.dim, self.dim, rngs=rngs)
        self.no_sink: bool      = config.no_sink

        self.W: nnx.Linear      = nnx.Linear(self.dim, self.dim, rngs=rngs)

        self.sliding_window: bool = config.sliding_window

        if self.sliding_window:
            table = jnp.arange(self.max_context)[:, None] - jnp.arange(self.max_context)[None, :]
            mask  = (table <= config.context_window)  & (table >= 0)
            self.window = jnp.where(mask, 0, -1e9)

        self.use_rotary: bool = config.use_rotary_pos
        if self.use_rotary:
            def __compute_angle(T:int, C:int) -> jnp.ndarray:
                P = jnp.arange(T)
                W = 1 / (1000 ** (jnp.arange(C//2) / C))
                degree = jnp.einsum('i,j->ij', P, W)[None, None, None, :, :]
                return degree
            
        self.attn_dropout: nnx.Dropout  = nnx.Dropout(config.dropout_rate, rngs=rngs)
        self.resid_dropout: nnx.Dropout = nnx.Dropout(config.dropout_rate, rngs=rngs)

        self.down_q: nnx.Linear  = nnx.Linear(config.dim, config.down_dim_q, rngs=rngs)
        self.down_kv: nnx.Linear = nnx.Linear(config.dim, config.down_dim_kv, rngs=rngs)

        self.up_q: nnx.Linear  = nnx.Linear(config.down_dim_q,  config.head_size * config.n_heads, rngs=rngs)
        self.up_k: nnx.Linear  = nnx.Linear(config.down_dim_kv, config.head_size * config.kv_heads, rngs=rngs)
        self.up_v: nnx.Linear  = nnx.Linear(config.down_dim_kv, config.head_size * config.kv_heads, rngs=rngs)

        self.down_dim_q: int  = config.down_dim_q
        self.down_dim_kv: int = config.down_dim_kv

        self.mla: bool       = config.mla
        self.inference: bool = config.inference

        self.rope_dim: int   = config.rope_dim if hasattr(config, 'rope_dim') else self.head_size // 2

        if self.mla:
            self.q_pe: nnx.Linear = nnx.Linear(config.dim, self.rope_dim, rngs=rngs)
            self.k_pe: nnx.Linear = nnx.Linear(config.dim, self.rope_dim, rngs=rngs)

        rope_dim_size           = self.head_size if self.mla is False else self.rope_dim
        self.angle: jnp.ndarray = __compute_angle(self.max_context, rope_dim_size) 


    def __apply_rotation(self, x: jnp.ndarray, cache_index: int) -> jnp.ndarray:
        T = x.shape[3]
        odd     = x[:, :, :, :, 0::2]
        even    = x[:, :, :, :, 1::2]

        angle   = jax.lax.dynamic_slice_in_dim(
                self.angle, 
                start_index=cache_index, 
                slice_size=T, 
                axis=3
            )
        x_odd  = jax.lax.cos(angle) * odd - jax.lax.sin(angle) * even
        x_even = jax.lax.sin(angle) * odd + jax.lax.cos(angle) * even

        y = jnp.stack([x_even, x_odd], axis=-1).reshape(x.shape)
        return y
    
    
    def _compute_cache(self, kv_cache, cache_index, B, T, k=None, v=None, c_kv=None, k_rope=None):
        if self.mla is False:
            if kv_cache == (None, None):  
                k_cache = jnp.zeros((B, self.kv_heads, 1, self.max_context, self.head_size), dtype=k.dtype)
                v_cache = jnp.zeros((B, self.kv_heads, 1, self.max_context, self.head_size), dtype=v.dtype)
                k_cache, v_cache = k_cache.at[:, :, :, :T, :].set(k), v_cache.at[:, :, :, :T, :].set(v)
            else:
                k_cache, v_cache = map(lambda x, y, index: jax.lax.dynamic_update_slice(x, y, (0, 0, 0, index, 0)), 
                                            (kv_cache[0], kv_cache[1]), (k, v), (cache_index, cache_index))

                
            kv_cache = (k_cache, v_cache)
            k, v     = k_cache, v_cache
            k_rope_cache = None
        else:
            if kv_cache == (None, None):  
                c_cache = jnp.zeros((B, self.max_context, self.down_dim_kv), dtype=c_kv.dtype)
                c_cache = c_cache.at[:, :T, :].set(c_kv)
                k_rope_cache = jnp.zeros((B, 1, 1, self.max_context, self.rope_dim), dtype=c_kv.dtype)
                k_rope_cache = k_rope_cache.at[:, :, :, :T, :].set(k_rope)
            else:
                c_cache = jax.lax.dynamic_update_slice(kv_cache[0], c_kv, (0, cache_index, 0))
                k_rope_cache = jax.lax.dynamic_update_slice(kv_cache[1], k_rope, (0, 0, 0, cache_index, 0))

            kv_cache = (c_cache, k_rope_cache)
            k, v = c_cache, c_cache

        return kv_cache, k_rope_cache, k,v
    

    def reshape_head(self, B, T, q, k, v):
        reshaping = lambda x, n_heads: jnp.reshape(x, (B, T, n_heads, self.head_size))

        q = reshaping(q, self.n_heads).reshape(B, T, self.kv_heads, 
                                                self.n_heads // self.kv_heads, self.head_size)
            
        k, v = map(reshaping, (k, v), (self.kv_heads, self.kv_heads))

        k, v = map(lambda x: jnp.expand_dims(x, axis=3), (k, v))

        permute = lambda x: jnp.transpose(x, (0, 2, 3, 1, 4))

        q, k, v = map(permute, (q, k, v))
        return q,k,v
    
    def _compute_attention(self, use_cache, kv_cache, cache_index, B, T, q, k, v):
        q, k, v = self.reshape_head(B, T, q, k, v)

        if self.use_rotary:
            q, k = map(self.__apply_rotation, (q, k), (cache_index, cache_index))

        if use_cache:
            kv_cache, _, k, v = self._compute_cache(kv_cache, cache_index, B, T, k, v, c_kv=None, k_rope=None)

        k = jnp.swapaxes(k, -2, -1)
        attn = q @ k / math.sqrt(self.head_size)
        return kv_cache, q, k, v, attn


    def __call__(self, x: jnp.ndarray, 
                 use_cache: bool, 
                 kv_cache: tuple, 
                 cache_index:int,
                 deterministic: bool = False) -> jnp.ndarray:
        
        B, T, _ = x.shape
        assert T <= self.max_context, "Sequence too Long"

        if self.mla is True:

            q = self.down_q(x)     
            c_kv = self.down_kv(x)

            if self.use_rotary:
                q_rope = self.q_pe(x)[:, None, None, :, :]
                k_rope = self.k_pe(x)[:, None, None, :, :]

                q_rope, k_rope = map(self.__apply_rotation, (q_rope, k_rope), (cache_index, cache_index))


            if self.inference is False:
                q = self.up_q(q)
                k, v = map(lambda func, vector: func(vector), (self.up_k, self.up_v), (c_kv, c_kv))

                q, k, v = self.reshape_head(B, T, q, k, v)

                q_rope_ext, k_rope_ext = map(lambda x, m: jnp.broadcast_to(x, m.shape[:-1] + (self.rope_dim,)),
                           (q_rope, k_rope), (q, k))
                
                q_full, k_full = map(lambda x, y: jnp.concatenate([x, y], axis=-1), (q, k), (q_rope_ext, k_rope_ext))

                k_full = jnp.swapaxes(k_full, -2, -1)

                attn   = q_full @ k_full / math.sqrt(self.head_size + self.rope_dim)

            else:
                if use_cache:
                    kv_cache, k_rope, k, v = self._compute_cache(kv_cache, cache_index, B, T, k=None, v=None, c_kv=c_kv, k_rope=k_rope)


                q_proj    = self.up_q.kernel.reshape(self.down_dim_q, self.kv_heads, 
                                                self.n_heads // self.kv_heads, self.head_size)
                
                k_proj    = self.up_k.kernel.reshape(self.down_dim_kv, self.kv_heads, self.head_size)
                
                attn_proj = jnp.einsum('qngh, knh -> ngqk', q_proj, k_proj)

                attn_proj = jnp.einsum('btq, ngqk  -> btngk', q, attn_proj)

                attn      = jnp.einsum('btngk, bsk -> bngts', attn_proj, k) 

                attn_rope = q_rope @ jnp.swapaxes(k_rope, -2, -1)

                attn      = (attn + attn_rope) / math.sqrt(self.head_size + self.rope_dim)


        elif self.mla is False:
            qkv = self.qkv(x)
            q_size  = self.dim
            kv_size = self.kv_heads * self.head_size

            q, k, v = jax.lax.split(
                qkv,
                (q_size, kv_size, kv_size),
                axis=-1
            )  

            kv_cache, q, k, v, attn = self._compute_attention(use_cache, kv_cache, cache_index, B, T, q, k, v)

        mask = jax.lax.dynamic_slice_in_dim(operand=self.tril, 
                                            start_index=cache_index,
                                            slice_size=T, 
                                            axis=0) 
        trilled = jnp.where(mask, 0.0, -1e9)

        attn = attn + trilled

        if self.sliding_window:
            attn = attn + jax.lax.dynamic_slice_in_dim(operand=self.window,
                                                start_index=cache_index,
                                                slice_size=T,
                                                axis=0)
        causal_attn = jax.nn.softmax(attn)
        causal_attn = self.attn_dropout(causal_attn, deterministic=deterministic)

        
        if self.inference is True and self.mla is True:

            L = jnp.einsum('bngts, bsd -> bngtd', causal_attn, v)
            
            W_v = self.up_v.kernel.reshape(self.down_dim_kv, self.kv_heads, self.head_size)

            if self.no_sink:

                y_heads = jnp.einsum('bngtd, dnh -> bngth', L, W_v)
                y = jnp.transpose(y_heads, (0, 3, 1, 2, 4)).reshape(B, T, self.dim)
                
                y = y * jax.nn.sigmoid(self.W(x))
                out = self.o_proj(y)
            else:
                W_o = self.o_proj.kernel.reshape(self.kv_heads, self.n_heads // self.kv_heads, 
                                                 self.head_size, self.dim)
                
                W_vo = jnp.einsum('dnh, nghc -> dngc', W_v, W_o)
                
                out = jnp.einsum('bngtd, dngc -> btc', L, W_vo)

        else:
            y = causal_attn @ v
            y = jnp.transpose(y, (0, 3, 1, 2, 4)).reshape(B, T, self.dim)

            if self.no_sink:
                y = y * jax.nn.sigmoid(self.W(x))

            out = self.o_proj(y)

        out = self.resid_dropout(out, deterministic=deterministic)
        return out, kv_cache






In [43]:
(nnx.Linear(4, 10, rngs=rngs).kernel @ nnx.Linear(10, 10, rngs=rngs).kernel).shape

(4, 10)

In [37]:
import torch

In [47]:
mask = torch.arange(10).unsqueeze(0) - torch.arange(10).unsqueeze(1)
print(mask)
mask = torch.where((mask<=0) & (mask>=-4), 0, 1)
print(mask)

tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9],
        [-1,  0,  1,  2,  3,  4,  5,  6,  7,  8],
        [-2, -1,  0,  1,  2,  3,  4,  5,  6,  7],
        [-3, -2, -1,  0,  1,  2,  3,  4,  5,  6],
        [-4, -3, -2, -1,  0,  1,  2,  3,  4,  5],
        [-5, -4, -3, -2, -1,  0,  1,  2,  3,  4],
        [-6, -5, -4, -3, -2, -1,  0,  1,  2,  3],
        [-7, -6, -5, -4, -3, -2, -1,  0,  1,  2],
        [-8, -7, -6, -5, -4, -3, -2, -1,  0,  1],
        [-9, -8, -7, -6, -5, -4, -3, -2, -1,  0]])
tensor([[0, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 1, 1, 1, 1, 1],
        [1, 0, 0, 0, 0, 0, 1, 1, 1, 1],
        [1, 1, 0, 0, 0, 0, 0, 1, 1, 1],
        [1, 1, 1, 0, 0, 0, 0, 0, 1, 1],
        [1, 1, 1, 1, 0, 0, 0, 0, 0, 1],
        [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]])


In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from dataclasses import dataclass
import math

@dataclass
class Config:
    dim: int = 256
    head_size: int = 8
    n_heads: int = 1
    context_size: int = 200
    kv_heads: int = 0
    sliding_w: int = 32

    def __post_init__(self):
        self.n_heads = self.dim // self.head_size
        self.kv_heads = self.n_heads // 4


class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.head_size = config.head_size
        self.dim = config.dim
        self.n_heads = config.n_heads
        self.kv_heads = config.kv_heads
        self.qkv = nn.Linear(config.dim, (self.n_heads + 2 * self.kv_heads) * self.head_size,bias=False)
        self.context_size = config.context_size

        self.o_proj = nn.Linear(config.dim, config.dim)
        trilled = torch.tril(torch.ones(self.context_size, self.context_size))
        self.register_buffer('tril', trilled)

        slinding_window = torch.arange(self.context_size).unsqueeze(0)-torch.arange(self.context_size).unsqueeze(1)
        mask = torch.where((slinding_window<=0) & (slinding_window>=-config.sliding_w), 0, -1e9)

        self.register_buffer('sliding_window_mask', mask)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B,T,C = x.size()

        q, k, v = self.qkv(x).split([self.head_size*self.n_heads, 
                                     self.kv_heads*self.head_size, 
                                     self.kv_heads*self.head_size], dim=-1)
        

        q, k, v = map(lambda x, g: x.view(B, T, self.kv_heads, g, self.head_size), (q, k, v), (self.n_heads//self.kv_heads, 1, 1))


        q, k, v = map(lambda x: x.permute(0, 2, 3, 1, 4), (q, k, v))

        attn = q @ k.transpose(-1, -2) / math.sqrt(self.head_size)

        causal = attn.masked_fill((self.tril[:T, :T] == 0), float('-inf'))

        causal = causal + self.sliding_window_mask[:T, :T]

        causal = F.softmax(causal, dim=-1)

        y = (causal @ v).permute(0, 3, 1, 2, 4)

        out = self.o_proj(y.contiguous().view(B, T, -1))

        return out.size()

        


config = Config()

print(config.kv_heads)
attn   = Attention(config)

B, T, C = 4, 100, 256
x = torch.rand((B,T,C))

attn(x)

8


torch.Size([4, 100, 256])

In [13]:
torch.arange(10).unsqueeze(0) - torch.arange(10).unsqueeze(1)


tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9],
        [-1,  0,  1,  2,  3,  4,  5,  6,  7,  8],
        [-2, -1,  0,  1,  2,  3,  4,  5,  6,  7],
        [-3, -2, -1,  0,  1,  2,  3,  4,  5,  6],
        [-4, -3, -2, -1,  0,  1,  2,  3,  4,  5],
        [-5, -4, -3, -2, -1,  0,  1,  2,  3,  4],
        [-6, -5, -4, -3, -2, -1,  0,  1,  2,  3],
        [-7, -6, -5, -4, -3, -2, -1,  0,  1,  2],
        [-8, -7, -6, -5, -4, -3, -2, -1,  0,  1],
        [-9, -8, -7, -6, -5, -4, -3, -2, -1,  0]])

In [ ]:
import torch
from torch import nn 
from torch.nn import functional as F
from dataclasses import dataclass
import math

@dataclass
class Config:
    dim: int = 256
    head_size: int = 8
    n_heads: int = 16
    kv_heads: int = 8
    context_size = 200
    window = 10
    def __post_init__(self):
        self.n_heads = self.dim // self.head_size
        self.kv_heads = self.n_heads // 4


class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.head_size = config.head_size
        self.context_size = config.context_size
        self.n_heads = config.n_heads
        self.kv_heads = config.kv_heads
        self.dim = config.dim
        self.qkv = nn.Linear(self.dim, self.head_size * (self.n_heads + 2*self.kv_heads))
        self.o_proj = nn.Linear(self.dim, self.dim)
        self.register_buffer('tril', torch.tril(torch.ones(self.context_size, self.context_size)))
        
        slid = torch.arange(self.context_size).unsqueeze(0) - torch.arange(self.context_size).unsqueeze(1)

        sliding_window = torch.where((slid<=0) & (slid>-config.window), 0, -1e9)
        self.register_buffer('sliding_window', sliding_window)

    def forward(self, x):
        B, T, C = x.size()

        q, k, v = self.qkv(x).split([self.head_size * self.n_heads, 
                                 self.head_size * self.kv_heads, 
                                 self.head_size * self.kv_heads], dim = -1)
        
        g_q = self.n_heads // self.kv_heads
        q, k, v = map(lambda x, g: x.view(B, T, self.kv_heads, g, self.head_size),
                       (q, k, v),
                       (g_q, 1, 1))
        
        q, k, v = map(lambda x: x.permute(0, 2, 3, 1, 4), (q, k, v))

        attn = q @ k.transpose(-1, -2) / math.sqrt(self.head_size)

        attn = attn.masked_fill((self.tril[:T, :T] == 0), float('-inf'))
        
        attn = attn + self.sliding_window[:T, :T]

        attn = F.softmax(attn, dim=-1)

        y = attn @ v

        y = y.permute(0, 3, 1, 2, 4).contiguous().view(B, T, -1)

        return self.o_proj(y)


    
config = Config()
attn = Attention(config)
B, T, C = 4, 100, 256
x = torch.rand((B, T, C))

attn(x)

tensor([[[ 6.4286e-02,  8.7461e-02, -1.4059e-01,  ...,  3.0132e-01,
           1.6058e-01, -2.0028e-02],
         [-5.3892e-03,  4.5253e-02, -8.3433e-02,  ...,  3.3737e-01,
           1.9098e-01, -1.0513e-01],
         [ 1.8138e-02,  5.6063e-02, -1.0561e-01,  ...,  3.1685e-01,
           1.5358e-01, -5.5040e-02],
         ...,
         [ 1.1665e-02,  3.1051e-02, -1.4203e-01,  ...,  3.1339e-01,
           1.8269e-01, -4.0823e-03],
         [ 1.1677e-02,  3.0017e-02, -1.4290e-01,  ...,  3.1359e-01,
           1.8308e-01, -3.7964e-03],
         [ 1.1457e-02,  2.9499e-02, -1.4263e-01,  ...,  3.1475e-01,
           1.8699e-01, -5.0679e-03]],

        [[ 7.7102e-02,  1.4943e-01, -2.0806e-01,  ...,  3.5124e-01,
           2.9903e-01,  1.6678e-01],
         [ 9.2942e-02,  1.7802e-01, -1.2620e-01,  ...,  3.5460e-01,
           2.2903e-01,  1.6974e-01],
         [ 4.4773e-02,  1.0261e-01, -1.0653e-01,  ...,  3.4482e-01,
           1.8552e-01,  1.3714e-01],
         ...,
         [ 7.0034e-03,  3

In [18]:
import torch
from torch import nn
from torch.nn import functional as F
from dataclasses import dataclass
from math import sqrt

@dataclass
class Config:
    dim: int = 128
    n_heads: int = 4
    kv_heads: int = 4
    head_size: int = 16
    context_size: int = 200
    W: int = 10
    n_experts: int = 8
    k: int = 2
    expansion: int = 2
    def __post_init__(self):
        self.n_heads = self.dim // self.head_size
        self.kv_heads = self.n_heads // 4

class Attention(nn.Module):
    def __init__(self, config: Config):
        super().__init__()
        self.dim = config.dim
        self.head_size = config.head_size
        self.kv_heads  = config.kv_heads
        self.head_size = config.head_size
        self.n_heads   = config.n_heads
        self.qkv = nn.Linear(self.dim, self.head_size * (config.n_heads + 2* self.kv_heads))

        self.o_proj = nn.Linear(self.dim, self.dim)
        self.register_buffer('tril', torch.tril(torch.ones(config.context_size, config.context_size)))

        sw = torch.arange(config.context_size).unsqueeze(0) - torch.arange(config.context_size).unsqueeze(1)
        mask = torch.where(((sw <=0) & (sw>=-config.W)), 0, -1e9)
        self.register_buffer('sliding_window', mask)

    def forward(self, x:torch.Tensor) -> torch.Tensor:
        B, T, C = x.size()
        q, k, v = self.qkv(x).split([self.head_size*self.n_heads,
                              self.head_size*self.kv_heads,
                              self.head_size*self.kv_heads], dim=-1)  #B, T, C
        g = self.n_heads // self.kv_heads
        q, k, v = map(lambda x, g: x.view(B, T, self.kv_heads, g, self.head_size),
                      (q, k, v),
                      (g, 1, 1))  #B, T, n, g, h
        
        q, k, v = map(lambda x: x.permute(0, 2, 3, 1, 4), (q, k, v))

        attn = q @ k.transpose(-2, -1) / sqrt(self.head_size)

        attn = attn.masked_fill((self.tril[:T, :T]==0), float('-inf'))

        attn = attn + self.sliding_window[:T, :T]

        attn = F.softmax(attn, dim=-1)

        y    = attn @ v

        y    = y.permute(0, 3, 1, 2, 4).contiguous().view(B, T, C)

        return self.o_proj(y)

class MoE(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.experts = nn.ModuleList([MLP(config) for _ in range(config.n_experts)])
        self.proj    = nn.Linear(config.dim, config.n_experts)
        self.k: int = config.k
        self.n_experts: int = config.n_experts

    def forward(self, x):
        logs = self.proj(x)
        tot_probs = F.softmax(logs, dim=-1)
        probs, indices = torch.topk(tot_probs, self.k, dim=-1)

        tot_probs = tot_probs.view(B*T, -1).mean(dim=0)

        freqs     = F.one_hot(indices.view(B*T, -1), num_classes=self.n_experts).float().sum(dim=1).mean(dim=0)

        loss = (freqs * tot_probs).sum() * self.n_experts

        probs = probs / probs.sum(dim=-1, keepdim=True)

        y = torch.zeros_like(x)

        for i in range(self.n_experts):
            mask = (indices == i)
            if not mask.any():
                pass

            chosens = mask.any(dim=-1) #, keepdim=True)
            weights = torch.where((mask==True), probs, 0).sum(dim=-1, keepdim=True)
            y[chosens] += weights[chosens] * self.experts[i](x[chosens])
            
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.up_proj = nn.Linear(config.dim, config.dim*config.expansion)
        self.gate    = nn.GELU()
        self.down_proj = nn.Linear(config.dim*config.expansion, config.dim)

        self.mlp = nn.Sequential(self.up_proj, self.gate, self.down_proj)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.mlp(x)
    
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attention = Attention(config)
        self.mlp       = MoE(config)
        self.ln1       = nn.LayerNorm(config.dim)
        self.ln2       = nn.LayerNorm(config.dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attention(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

config = Config()
B, T, C = 4, 100, 128
x = torch.rand((B, T, C))

block = Block(config)

block(x)


tensor([[[ 0.4318,  0.2786,  0.9979,  ...,  0.8264,  0.1967,  0.0124],
         [ 0.5572,  0.6904,  0.8682,  ...,  0.6863,  1.1069,  0.5598],
         [ 0.4544,  0.5421,  1.2063,  ...,  0.5025,  0.9069,  0.6793],
         ...,
         [ 0.1333,  0.6631,  0.6953,  ...,  0.5627, -0.1848,  0.7202],
         [ 0.5900,  0.4113,  0.4563,  ...,  0.7813,  0.6406,  0.7565],
         [ 0.3086,  0.5429,  0.8645,  ...,  0.8920,  0.1345,  0.2962]],

        [[ 0.2413,  0.6250,  1.1848,  ...,  0.8340, -0.2123,  1.2012],
         [ 0.2906,  0.2377,  0.7228,  ...,  0.3749,  0.5981,  0.4395],
         [-0.2777, -0.0405,  0.6860,  ...,  0.7132,  0.7901,  0.3306],
         ...,
         [ 0.8028,  0.3856,  0.7906,  ...,  0.8814,  0.9671,  0.9341],
         [ 0.4969,  0.4357,  0.7126,  ...,  0.7399,  0.9510,  0.6864],
         [ 0.6527,  0.6969,  0.4625,  ...,  0.9977,  0.5848,  1.0367]],

        [[ 0.1308, -0.6824,  0.0086,  ...,  0.2025, -0.1293,  1.1319],
         [ 0.8329,  0.0029,  0.1944,  ...,  0

In [2]:
import torch 
from torch import nn
from torch.nn import functional as F 
from dataclasses import dataclass
from math import sqrt

@dataclass
class Config:
    dim: int = 128
    head_size: int = 8
    context_size: int = 200
    kv_heads: int = 4
    window: int = 10

    def __post_init__(self):
        self.n_heads = self.dim // self.head_size
        self.kv_heads = self.n_heads // 4

class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.dim = config.dim
        self.head_size = config.head_size
        self.context_size = config.context_size
        self.n_heads  = config.n_heads
        self.kv_heads = config.kv_heads

        self.register_buffer('tril', torch.tril(torch.ones(config.context_size, config.context_size)))

        self.qkv = nn.Linear(config.dim, config.head_size * (config.n_heads + 2*config.kv_heads))
        self.o_proj = nn.Linear(config.dim, config.dim)
        
        sw: torch.Tensor = torch.arange(config.context_size).unsqueeze(0) - torch.arange(config.context_size).unsqueeze(1)
        mask: torch.Tensor = torch.where(((sw<=0) & (sw>=-config.window)), 0, 1e-9)
        self.register_buffer('sliding_window', mask)


    def forward(self, x):
        B, T, C = x.size()

        h, n, d, g = self.head_size, self.n_heads, self.kv_heads, self.n_heads // self.kv_heads
        q, k, v = self.qkv(x).split([h*n, h*d, h*d], dim=-1)                 
        
        q, k, v = map(lambda x, gr: x.view(B, T, self.kv_heads, gr, self.head_size), (q, k, v), (g, 1, 1))

        q, k, v = map(lambda x: x.permute(0, 2, 3, 1, 4), (q, k, v))

        attn = q @ k.transpose(-2, -1) / sqrt(self.head_size)

        attn = attn.masked_fill((self.tril[:T, :T]==0), float('-inf'))

        attn = attn + self.sliding_window[:T, :T]
        attn = F.softmax(attn, dim=-1)

        y    = attn @ v

        y    = y.permute(0, 3, 1, 2, 4).contiguous().view(B, T, C)

        return self.o_proj(y)



class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.up: nn.Linear = nn.Linear(config.dim, config.expansion* config.dim)
        self.gate: nn.GELU = nn.GELU()
        self.down: nn.Linear = nn.Linear(config.dim*config.expansion, config.dim)
        self.mlp : nn.Sequential = nn.Sequential(self.up, self.gate, self.down)

    def forward(self, x):
        return self.mlp(x)
    


config = Config()

B, T, C = 4, 200, 128

x = torch.rand((B, T, C))

attn = Attention(config)

attn(x)

tensor([[[-0.1595, -0.1016,  0.2358,  ..., -0.0328,  0.0795, -0.0528],
         [-0.1245, -0.0827,  0.2411,  ..., -0.1007,  0.1957, -0.0408],
         [-0.0870, -0.0527,  0.1986,  ..., -0.0850,  0.1963, -0.0209],
         ...,
         [-0.1592, -0.0163,  0.0865,  ..., -0.0458,  0.1112,  0.0685],
         [-0.1593, -0.0160,  0.0868,  ..., -0.0460,  0.1121,  0.0680],
         [-0.1597, -0.0153,  0.0869,  ..., -0.0454,  0.1112,  0.0678]],

        [[-0.2758,  0.0399,  0.2224,  ..., -0.0106,  0.1451,  0.0728],
         [-0.2410, -0.0784,  0.1833,  ..., -0.0186,  0.1602,  0.0374],
         [-0.2358, -0.0866,  0.1270,  ..., -0.0611,  0.1645,  0.0673],
         ...,
         [-0.1569, -0.0189,  0.0810,  ..., -0.0540,  0.1167,  0.0758],
         [-0.1569, -0.0186,  0.0805,  ..., -0.0535,  0.1165,  0.0757],
         [-0.1569, -0.0194,  0.0811,  ..., -0.0538,  0.1172,  0.0761]],

        [[-0.0867,  0.0525, -0.0090,  ..., -0.0177,  0.0377,  0.0197],
         [-0.1553,  0.0215,  0.0785,  ...,  0

In [10]:
import torch
from torch import nn
from torch.nn import functional as F
from math import sqrt

class Attention(nn.Module):
    def __init__(self, config:Config):
        super().__init__()
        self.dim = config.dim
        self.head_size = config.head_size
        self.n_heads = config.n_heads
        self.context_size = config.context_size
        self.qkv = nn.Linear(config.dim, config.head_size*(self.n_heads + 2*config.kv_heads))
        self.o_proj = nn.Linear(config.dim, config.dim)
        self.kv_heads = config.kv_heads
        self.register_buffer('tril', torch.tril(torch.ones(config.context_size, config.context_size)))

        sw = torch.arange(config.context_size).unsqueeze(0) - torch.arange(config.context_size).unsqueeze(1)
        mask = torch.where(((sw<=0) & (sw>=-config.window)), 0, float('-inf'))
        self.register_buffer('sliding_window', mask)

    def forward(self, x):
        B, T, C = x.size()
        h, n, c, g = self.head_size, self.n_heads, self.kv_heads, self.n_heads // self.kv_heads

        q, k, v = self.qkv(x).split([h*n, h*c, h*c], dim=-1)

        q, k, v = map(lambda x, gr: x.view(B, T, self.kv_heads, gr, self.head_size), (q, k, v), (g, 1, 1))

        q, k, v = map(lambda x: x.permute(0, 2, 3, 1, 4), (q, k, v))

        print(q.shape, k.shape, v.shape)

        attn = q @ k.transpose(-2, -1) / sqrt(self.head_size)

        attn = attn.masked_fill((self.tril[:T, :T] == 0), float('-inf'))

        attn = attn + self.sliding_window[:T, :T]

        attn = F.softmax(attn, dim=-1)


        y = attn @ v

        y = y.permute(0, 3, 1, 2, 4).contiguous().view(B, T, C)

        print(y.shape)
        return self.o_proj(y)


B, T, C = 4, 200, 128

x = torch.rand((B, T, C))

attn = Attention(config)

attn(x)

torch.Size([4, 4, 4, 200, 8]) torch.Size([4, 4, 1, 200, 8]) torch.Size([4, 4, 1, 200, 8])
torch.Size([4, 200, 128])


tensor([[[ 0.0753, -0.1195,  0.1971,  ...,  0.0301, -0.0542, -0.0880],
         [ 0.0258, -0.1586,  0.0594,  ..., -0.0055, -0.0108, -0.1021],
         [ 0.0100, -0.1601,  0.0559,  ..., -0.0177,  0.0188, -0.1024],
         ...,
         [-0.0426, -0.2325,  0.0183,  ...,  0.0027, -0.0349, -0.0852],
         [-0.0334, -0.2240,  0.0327,  ...,  0.0205, -0.0261, -0.0832],
         [-0.0160, -0.2197,  0.0334,  ...,  0.0183, -0.0269, -0.0706]],

        [[-0.0329, -0.1705, -0.0633,  ...,  0.1648,  0.1017, -0.0709],
         [-0.0603, -0.1606,  0.0618,  ...,  0.0665,  0.0090, -0.0626],
         [-0.0467, -0.1971,  0.0434,  ..., -0.0048, -0.0455, -0.1173],
         ...,
         [-0.0449, -0.1801,  0.0546,  ...,  0.0635, -0.0211, -0.0905],
         [-0.0441, -0.1745,  0.0665,  ...,  0.0401, -0.0195, -0.0951],
         [-0.0197, -0.1835,  0.0597,  ...,  0.0193, -0.0146, -0.0936]],

        [[-0.0643, -0.2284, -0.0280,  ...,  0.0548,  0.0200, -0.1280],
         [-0.0236, -0.1621, -0.0145,  ...,  0